<a href="https://colab.research.google.com/github/vishaljoshi24/DungeonsAndDragonsTurnClassification/blob/Codex-Branch/Experiment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
print(sys.version)
print(sys.executable)

!{sys.executable} -m pip --version

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
/usr/bin/python3
pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)


In [2]:
!git clone -b Codex-Branch https://github.com/vishaljoshi24/DungeonsAndDragonsTurnClassification/
%cd /content/DungeonsAndDragonsTurnClassification

fatal: destination path 'DungeonsAndDragonsTurnClassification' already exists and is not an empty directory.
/content/DungeonsAndDragonsTurnClassification


In [3]:
import requests

response = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)
print(response.status_code)
print(response.json())

200
{'models': [{'name': 'qwen3:4b', 'model': 'qwen3:4b', 'modified_at': '2026-10-26T19:28:59.259701613Z', 'size': 2497293931, 'digest': '359d7dd4bcdab3d86b87d73ac27966f4dbb9f5efdfcc75d34a8764a09474fae7', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen3', 'families': ['qwen3'], 'parameter_size': '4.0B', 'quantization_level': 'Q4_K_M', 'context_length': 262144, 'embedding_length': 2560, 'runner': 'ggml'}, 'capabilities': ['completion', 'tools', 'thinking']}]}


In [4]:
import subprocess

result = subprocess.run(
    ["ollama", "pull", "qwen3:4b"],
    capture_output=False,
    text=True,
)

print("Download finished with exit code:", result.returncode)

Download finished with exit code: 0


In [5]:
import requests

response = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)
response.raise_for_status()

models = response.json()["models"]
for model in models:
    print(model["name"])

qwen3:4b


In [6]:
!pip install dspy==3.2.1

In [7]:
import argparse

In [8]:
from modules import(
    AgentProfile,
    BaselineConversation,
    configure_lm,
    BaselineChatbot
)

In [9]:
RAT_INFESTATION_SCENARIO = (
'This D&D short scenario specifically concerns a rat infestation. '
'It is set in a craft brewery and is in dire need of help. '
'The players in this scenario are Alice and Bob.'
'The non-player character is Charlie, and he is controlled by the Dungeon Master. '
'Alice and Bob are here to sort out a RAT INFESTATION in the brewery\'s BASEMENT. '
'For the duration of the scenario, only Alice, Bob and the brewery owner Charlie are in the brewery '
'At the beginning of this adventure Alice and Bob '
'meet in the main taproom of the brewery. These two adventurers '
'DO NOT know each other AT FIRST and need to get to know each other. '
'Charlie hands out pints of Ale to Alice and Bob as they get to know each other. '
'Alice and Bob have the following objectives: '
'1. Alice and Bob need to find out where the rats in the cellar were first seen coming from.'
'2. Alice and Bob need a map of the cellar.'
'3. Alice and Bob need a key to enter the cellar.'
'4. Alice and Bob should consult the map when planning their approach.'
'5. Alice and Bob need to discuss their formation when approaching the cellar.'
'6. Alice and Bob should discuss which resources they have which can help with their exploration of the cellar.'
'7. Alice and Bob should move down into the cellar based on the formation they discussed.'
'8. Alice and Bob should use the resources they discussed, to carry out their exploration.'
'9. Alice and Bob need to find signs of the infestation.'
)

MISSING_CHILD_SCENARIO = (
  'This D&D short campaign specifically concerns Diana, a magical walrus who has brought a group of animals to safety. '
  'She has brought the animals to the mountains, hoping to protect them from being used by an evil druid to wreak havoc on the local area. '
  'She requests the help of Alice and Bob to stem the threats against everyone. '
  'For the duration of this scenario, only Alice, Bob and Diana are in a cave in the mountains. '
  'At the beginning of this adventure Alice and Bob '
  'meet Diana at her cave in the mountains. The three adventurers '
  'DO NOT know each other AT FIRST and need to get to know each other.'
  'The cave in which the three characters meet has a river which flows out to the open air. '
  'Diana asks Alice and Bob to find her child, an awakened otter that has been spying on a nearby city. '
  'Alice and Bob should start by asking Diana some questions about the missing child. '
  'Alice and Bob should then come up with a strategy for their search of the child. '
  'Finally, Alice and Bob should explore the surrounding mountains based on their strategy. '
)

In [10]:
def build_conversation() -> BaselineConversation:
    alice = AgentProfile(
        name="Alice",
        character="Alice, a dwarf barbarian.",
        instructions=(
        'The instructions for how to play the role of a D&D player are as '
        'follows. This is a short scenario in which you '
        f'play the role of a character named Alice. This scenario '
        'is structured as a Dungeons & Dragons game. '
        'The goal is to be consistent, but creative. It is '
        'important to play the role of a Dungeons & Dragons player as '
        'accurately as possible, i.e., by responding in ways that you think '
        'it is likely a player would respond, and taking '
        'into account all information that you have. '
        'It is important that you collaborate with the  '
        'other player, on the task at hand and listen to the Dungeon Master\'s '
        'insturctions.'
        'Do not be verbose. '
        'Do not repeat yourself. '
        'Always use first-person limited perspective.'
        ),
        inventory = [
                "One set of common clothes",
                "Two daggers",
                "One axe",
                "50 feet of rope",
                "One tinderbox",
                "One torch",
            ],
        abilities = {""},
        skills = {""},
    )
    bob = AgentProfile(
        name="Bob",
        character="Bob, a wizard.",
        instructions=(
        'The instructions for how to play the role of a D&D player are as '
        'follows. This is a short scenario in which you '
        'play the role of a character named Bob. This scenario '
        'is structured as a Dungeons & Dragons game. '
        'The goal is to be consistent, but creative. It is '
        'important to play the role of a Dungeons & Dragons player as '
        'accurately as possible, i.e., by responding in ways that you think '
        'it is likely a player would respond, and taking '
        'into account all information that you have. '
        'It is important that you collaborate with the  '
        'other player, on the task at hand and listen to the Dungeon Master\'s '
        'insturctions.'
        'Do not be verbose. '
        'Do not repeat yourself. '
        'Always use first-person limited perspective.'
        ),
        inventory = [
                "One Wizard's Staff",
                "One can of oil",
                "One tinderbox",
                "Thunderwave spell: You unleash a wave of thunderous energy.",
                "Command spell: You speak a one-word command to a creature you can see within range."
            ],
        abilities = {""},
        skills = {""},
    )
    dungeon_master = AgentProfile(
          name="Dungeon Master",
          character="The Dungeon Master, who also plays the role of Charlie: the owner of the Wizards Tower Brewing Company.",
          instructions= (
          'This is a tabletop role-playing game: Dungeons & Dragons. You are the Dungeon Master.'
          'You will describe the current situation to the players in the game and then on the basis '
          'of what you tell them they will suggest actions for the character they control. '
          'You will then decide if the action is valid based on Dungeons & Dragons 5th Edition rules. '
          'Aside from you, each other player controls just one character. '
          'If any of the players deviates dramatically from the scenario your response should attempt to re-orient the scenario. '
          'You are the Dungeon Master so you control any non-player characters or adversaries. '
          'You can answer player\'s questions and give them any items they need, but you should '
          'not get involved with the task directly. '
          'You will track the state of the world and keep it consistent as time  '
          'passes in the simulation and the players take actions and change things in their world. '
          'Remember that this is a game. It should be fun for the players. '
          'You should use second-person perspective, when speaking directly to the players. '
          'You should use first-person limited perspective when role-playing as non-player characters and adversaries.'
          'Do not be verbose. Do not repeat yourself.'
          ),
          inventory = [""],
          abilities = {""},
          skills = {""},
    )

    return BaselineConversation(
        agent_a=alice,
        agent_b=bob,
        agent_c=dungeon_master,
        scenario=RAT_INFESTATION_SCENARIO,
        chatbot_a=BaselineChatbot(),
        chatbot_b=BaselineChatbot(),
        chatbot_c=BaselineChatbot()
    )

In [11]:
parser = argparse.ArgumentParser(description="Run three D&D DSPy agents together.")

In [12]:
parser.add_argument(
    "--opening-message",
    default="You find yourselves in the main taproom of the brewery, questioning Charlie the brewery owner.",
    help="The first turn spoken by the Dungeon Master.",
)

_StoreAction(option_strings=['--opening-message'], dest='opening_message', nargs=None, const=None, default='You find yourselves in the main taproom of the brewery, questioning Charlie the brewery owner.', type=None, choices=None, required=False, help='The first turn spoken by the Dungeon Master.', metavar=None)

In [13]:
parser.add_argument(
    "--rounds",
    type=int,
    default=30,
    help="Number of replies to generate after the opening message.",
)

_StoreAction(option_strings=['--rounds'], dest='rounds', nargs=None, const=None, default=30, type=<class 'int'>, choices=None, required=False, help='Number of replies to generate after the opening message.', metavar=None)

In [14]:
parser.add_argument("--model", default="ollama_chat/qwen3:4b")

_StoreAction(option_strings=['--model'], dest='model', nargs=None, const=None, default='ollama_chat/qwen3:4b', type=None, choices=None, required=False, help=None, metavar=None)

In [15]:
parser.add_argument("--api-base", default="http://localhost:11434")

_StoreAction(option_strings=['--api-base'], dest='api_base', nargs=None, const=None, default='http://localhost:11434', type=None, choices=None, required=False, help=None, metavar=None)

In [16]:
parser.add_argument("--max-tokens", type=int, default=8192)

_StoreAction(option_strings=['--max-tokens'], dest='max_tokens', nargs=None, const=None, default=8192, type=<class 'int'>, choices=None, required=False, help=None, metavar=None)

In [17]:
parser.add_argument(
    "--cache",
    action="store_true",
    help="Enable LM response caching (disabled by default)",
    default=False
)

_StoreTrueAction(option_strings=['--cache'], dest='cache', nargs=0, const=True, default=False, type=None, choices=None, required=False, help='Enable LM response caching (disabled by default)', metavar=None)

In [18]:
parser.add_argument("--think", action="store_true", default=False)


_StoreTrueAction(option_strings=['--think'], dest='think', nargs=0, const=True, default=False, type=None, choices=None, required=False, help=None, metavar=None)

In [19]:
args = parser.parse_args([])

In [20]:
configure_lm(model=args.model, api_base=args.api_base)

In [21]:
conversation = build_conversation()

In [22]:
for turn in conversation.run(args.opening_message, rounds=args.rounds):
    print(turn)

Dungeon Master: You find yourselves in the main taproom of the brewery, questioning Charlie the brewery owner.
Alice: Charlie, where did the rats first come from?
Bob: Need a map.
Dungeon Master: Charlie hands you a map of the cellar and a pint of ale.
Alice: Thanks, Charlie. Bob, check the map for rat origin.
Bob: West storage.
Dungeon Master: You both look at the map. Bob points to west storage. The cellar entrance is locked. Charlie: "Key is in taproom." What next?
Alice: Charlie, give us the key.
Bob: Got the key. Let's go.
Dungeon Master: You step into the cellar. Rats scuttle in the shadows.
Alice: Rats are swarming. Bob, stay close and move fast. I see a narrow path to the left.
Bob: Left. Stay close.
Dungeon Master: You find a narrow passage ahead. Rats scuttle in the shadows.
Alice: I go first.
Bob: I follow. Watch for rats.
Dungeon Master: Rats scuttle ahead.
Alice: I see a trail of droppings heading west.
Bob: West. I'll follow the trail.
Dungeon Master: You follow the trail

In [23]:
import dspy
dspy.inspect_history(n=10)





[2026-10-26T20:10:24.981673]

System message:

Your input fields are:
1. `instruct_prompt` (str): 
2. `scenario_context` (str): 
3. `user_input` (str):
Your output fields are:
1. `response` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## instruct_prompt ## ]]
{instruct_prompt}

[[ ## scenario_context ## ]]
{scenario_context}

[[ ## user_input ## ]]
{user_input}

[[ ## response ## ]]
{response}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Respond as a D&D player character using the supplied scenario and behaviour instructions.


User message:

[[ ## instruct_prompt ## ]]
You are playing a Dungeons & Dragons scene.
Character: Alice, a dwarf barbarian.
Instructions: The instructions for how to play the role of a D&D player are as follows. This is a short scenario in which you play the role of a character named Alice. This scenario is structured as a Dungeons & Dragons game. The goal is 